# Customer Churn Prediction - Feature Engineering

This notebook adds a small set of interpretable features from the existing customer attributes. The transformations are applied before the train/test split so the resulting columns can be reused consistently in later experiments.

In [ ]:
import pandas as pd
import numpy as np

data_path = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df.head()

## Create derived features

The new columns describe service count, average monthly spend over the customer's tenure, and whether the customer uses automatic payment. These are derived only from information already available in the dataset.

In [ ]:
service_columns = [
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

df["ServiceCount"] = (df[service_columns] == "Yes").sum(axis=1)
df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, np.nan)
df["AvgMonthlySpend"] = df["AvgMonthlySpend"].fillna(df["MonthlyCharges"])
df["AutoPayment"] = df["PaymentMethod"].str.contains(
    "automatic", case=False, na=False
).astype(int)

df[["ServiceCount", "AvgMonthlySpend", "AutoPayment"]].describe()

## Inspect the new features

In [ ]:
feature_columns = ["ServiceCount", "AvgMonthlySpend", "AutoPayment"]
print(df[feature_columns].isna().sum())
print(df[feature_columns].head(10))

## Prepare the modeling data

The engineered columns are added to the feature matrix while the customer identifier and target remain excluded.

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Total missing values:", X.isna().sum().sum())